In [1]:
!pip install torch_geometric

# ! pip install infomap networkx

!pip install pyg-lib -f https://data.pyg.org/whl/torch-2.6.0+cu124.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.5 MB/s eta 0:00:00a 0:00:01
Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 41.6 MB/s eta 0:00:0000:0100:01


In [2]:
import torch
print("Torch Version:", torch.__version__)
import numpy as np
import pandas as pd
from sklearn.decomposition import non_negative_factorization
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T
from collections import defaultdict
import numpy as np
import torch
import tqdm
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, to_hetero
from torch_geometric.loader import LinkNeighborLoader
from torch import Tensor
from networkx.algorithms.community.quality import modularity
from typing import Dict, List, Tuple
import torch.nn as nn
import numpy as np
from math import log2

Torch Version: 2.6.0+cu124


In [4]:
import pandas as pd

# Load data
df = pd.read_json("/kaggle/input/amazon-music-2018/Digital_Music.json", lines=True)

# Keep relevant columns and remove duplicates
# df = df[['user_id', 'parent_asin', 'text']].drop_duplicates()
df = df[['reviewerID', 'asin', 'reviewText']].drop_duplicates()

# Compute word counts and filter reviews
df['word_count'] = df['reviewText'].str.split().str.len()
df = df[df['word_count'] >= 10]

# Filter for users who have reviewed at least 10 different products
df = df.groupby('reviewerID').filter(lambda x: x['asin'].nunique() >= 10)

filtered_df = df
print(f"Filtered dataset size: {len(df)}")
print(f"Remaining unique users: {df['reviewerID'].nunique()}")
print(f"Remaining unique products: {df['asin'].nunique()}")

Filtered dataset size: 146057
Remaining unique users: 6921
Remaining unique products: 93774


In [3]:
import pandas as pd

# Load data
df = pd.read_json("/kaggle/input/amazon-music-2018/Digital_Music.json", lines=True)

# # Keep relevant columns and remove duplicates
# df = df[['user_id', 'parent_asin', 'text']].drop_duplicates()

# Compute word counts and filter reviews
df['word_count'] = df['reviewText'].str.split().str.len()
df = df[df['word_count'] >= 50]

# Filter products with >=5 interactions
min_interactions_product = 20

while True:
    product_counts = df.groupby('asin')['reviewerID'].nunique()
    valid_products = product_counts[product_counts >= min_interactions_product].index
    
    prev_len = len(df)
    df = df[df['asin'].isin(valid_products)]
    
    # stop when no more changes happen
    if len(df) == prev_len:
        break


min_interactions_user = 10

while True:
    user_counts = df.groupby('reviewerID')['asin'].nunique()
    valid_users = user_counts[user_counts >= min_interactions_user].index
    
    prev_len = len(df)
    df = df[df['reviewerID'].isin(valid_users)]
    
    # stop when no more changes happen
    if len(df) == prev_len:
        break

print(f"Filtered dataset size: {len(df)}")
print(f"Remaining unique users: {df['reviewerID'].nunique()}")
print(f"Remaining unique products: {df['asin'].nunique()}")

filtered_df = df[['reviewerID', 'asin']]


Filtered dataset size: 2027
Remaining unique users: 102
Remaining unique products: 663


In [5]:
filtered_df = df

In [6]:
unique_users = filtered_df['reviewerID'].unique()

user_id_map = {uid: idx for idx, uid in enumerate(unique_users)}

filtered_df['mapped_user_id'] = filtered_df['reviewerID'].map(user_id_map)
filtered_df['reviewText'] = filtered_df['reviewText'].fillna("").astype(str)

combined_df = (
    filtered_df.groupby('mapped_user_id')['reviewText']
    .apply(lambda texts: ". ".join(texts))
    .reset_index()
    .rename(columns={'reviewText': 'combined_text'})
)
combined_df

,mapped_user_id,combined_text
0,0,Keith Green / So you wanna go back to Egypt......
1,1,Don Francisco is one of the performers who had...
2,2,"John Michael Talbot's ""Master Collection Volum..."
3,3,"I have enjoyed these songs for decades, and ha..."
4,4,"Some may give this two cd set five stars, but ..."
...,...,...
6916,6916,Feeling Marta is a beautiful album interpreted...
6917,6917,All Your Heart is a classic love story. Reaga...
6918,6918,"As a diplomat throughout the world, Sossi has ..."
6919,6919,"1. NAGUAL (25'33"")\nFrom a deep rhythm evokin..."


In [7]:
combined_df.to_csv("/kaggle/working/combined_reviews.csv", index=False)

In [8]:
personality = pd.read_csv('/kaggle/input/music-personality/music_personality.csv')
personality

,mapped_user_id,Openness,Conscientiousness,Extraversion,Agreeableness,Neuroticism
0,0,9.2,5.4,3.9,9.0,1.9
1,1,8.4,2.9,2.8,2.3,3.1
2,2,4.2,0.8,1.7,3.0,1.9
3,3,8.1,3.4,1.4,7.6,2.9
4,4,9.0,6.6,2.7,7.9,5.3
...,...,...,...,...,...,...
1009,1009,7.9,5.3,3.3,6.0,4.4
1010,1010,9.3,1.9,2.2,3.1,2.7
1011,1011,8.5,4.7,4.2,8.1,2.3
1012,1012,6.4,3.8,7.0,7.8,6.6


In [9]:
num_users = 6921

user_ids = np.arange(num_users)

data = {
    "mappedUserID": user_ids,
    "openness": np.round(np.random.uniform(1, 10, num_users), 1),
    "agreeableness": np.round(np.random.uniform(1, 10, num_users), 1),
    "emotional_stability": np.round(np.random.uniform(1, 10, num_users), 1),
    "conscientiousness": np.round(np.random.uniform(1, 10, num_users), 1),
    "extraversion": np.round(np.random.uniform(1, 10, num_users), 1),
}

df1 = pd.DataFrame(data)
df1.head()

,mappedUserID,openness,agreeableness,emotional_stability,conscientiousness,extraversion
0,0,2.6,1.7,8.4,1.3,4.0
1,1,5.2,3.9,5.2,3.3,5.3
2,2,4.8,9.2,9.3,5.9,4.4
3,3,6.5,3.6,2.3,7.2,9.5
4,4,7.9,5.9,6.7,4.6,7.2


In [10]:
filtered_df = filtered_df[['reviewerID', 'asin', 'reviewText']]

unique_user_id = filtered_df['reviewerID'].unique()
unique_user_id = pd.DataFrame({
    'reviewerID': unique_user_id,
    'mappedUserID': pd.RangeIndex(len(unique_user_id))
})
print("Mapping of user IDs to consecutive values:")
print("==========================================")
print(unique_user_id.head(), "\n")

unique_product_id = filtered_df['asin'].unique()
unique_product_id = pd.DataFrame({
    'asin': unique_product_id,
    'mappedProductID': pd.RangeIndex(len(unique_product_id))
})
print("Mapping of product IDs to consecutive values:")
print("=============================================")
print(unique_product_id.head(), "\n")

ratings_user_id = pd.merge(
    filtered_df[['reviewerID']], unique_user_id,
    on='reviewerID', how='left'
)
ratings_user_id = torch.from_numpy(ratings_user_id['mappedUserID'].values)

ratings_product_id = pd.merge(
    filtered_df[['asin']], unique_product_id,
    on='asin', how='left'
)
ratings_product_id = torch.from_numpy(ratings_product_id['mappedProductID'].values)

edge_index_user_to_product = torch.stack([ratings_user_id, ratings_product_id], dim=0)

print("Final edge indices pointing from users to products:")
print("===================================================")
print(edge_index_user_to_product)


Mapping of user IDs to consecutive values:
       reviewerID  mappedUserID
0  A12R54MKO17TW0             0
1  A3SALRCJ8EJI83             1
2   A3FVAWZNKW9GX             2
3  A27SJD1VM73SMM             3
4  A1BVU7F2T8EUKS             4 

Mapping of product IDs to consecutive values:
         asin  mappedProductID
0  0001388703                0
1  0001527134                1
2  0001377647                2
3  0006920055                3
4  0830838015                4 

Final edge indices pointing from users to products:
tensor([[    0,     1,     2,  ...,  3713,   330,  5027],
        [    0,     1,     2,  ..., 93771, 93772, 93773]])


In [11]:
import numpy as np
from collections import defaultdict

seed = 42
rng = np.random.RandomState(seed)

interactions = edge_index_user_to_product.numpy().T  # shape (num_edges, 2)
# each row = [user_id, product_id]

min_interactions = 2  # we need at least 2 so that at least 1 stays in train

# Build per-user lists
user_to_items = defaultdict(list)
for u, i in interactions:
    user_to_items[int(u)].append(int(i))

train_edges = []
val_edges = []
test_edges = []

for u, items in user_to_items.items():
    n = len(items)
    if n < min_interactions:
        # not enough to hold out, keep all in train
        train_edges.extend([(u, i) for i in items])
        continue

    # pick 1 test edge
    test_i = rng.choice(items, size=1)[0]
    test_edges.append((u, test_i))

    remaining = [i for i in items if i != test_i]

    if len(remaining) >= 2:
        # pick 1 validation edge
        val_i = rng.choice(remaining, size=1)[0]
        val_edges.append((u, val_i))
        remaining = [i for i in remaining if i != val_i]

    # all others go to train
    train_edges.extend([(u, i) for i in remaining])

train_edges = np.array(train_edges, dtype=int)
val_edges = np.array(val_edges, dtype=int)
test_edges = np.array(test_edges, dtype=int)

print("train edges:", len(train_edges),
      "val edges:", len(val_edges),
      "test edges:", len(test_edges))


train edges: 132117 val edges: 6921 test edges: 6921


In [12]:
data = HeteroData()

data["user"].node_id = torch.arange(len(unique_user_id))
data["product"].node_id = torch.arange(len(unique_product_id))

data["user"].x = torch.tensor(df1.drop('mappedUserID',axis=1).values, dtype=torch.float32)
data["product"].x = None

edge_index = torch.tensor(train_edges.T, dtype=torch.long).contiguous() 
data["user", "rates", "product"].edge_index = edge_index

data = T.ToUndirected()(data)

In [13]:
from torch_geometric.loader import LinkNeighborLoader

# Use train_edges for supervision
edge_label_index = torch.tensor(train_edges.T, dtype=torch.long)
edge_label = torch.ones(edge_label_index.size(1), dtype=torch.float)

train_loader = LinkNeighborLoader(
    data=data,
    num_neighbors=[20, 10],
    neg_sampling_ratio=2.0,   # negatives generated on-the-fly
    edge_label_index=(("user", "rates", "product"), edge_label_index),
    edge_label=edge_label,
    batch_size=128,
    shuffle=True,
)


In [14]:
class GNN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(hidden_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
    def forward(self, x: Tensor, edge_index: Tensor) -> Tensor:
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

class Classifier(torch.nn.Module):
    def forward(self, x_user: Tensor, x_product: Tensor, edge_label_index: Tensor) -> Tensor:
        edge_feat_user = x_user[edge_label_index[0]]
        edge_feat_product = x_product[edge_label_index[1]]
        return (edge_feat_user * edge_feat_product).sum(dim=-1)

class PersonalityPredictor(torch.nn.Module):
    def __init__(self, in_channels, out_channels=5):
        super().__init__()
        self.lin1 = torch.nn.Linear(in_channels, in_channels)
        self.lin2 = torch.nn.Linear(in_channels, out_channels)

    def forward(self, x_user: Tensor) -> Tensor:
        x = F.relu(self.lin1(x_user))
        x = self.lin2(x) 
        return x

class Model_linear(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        # Since the dataset does not come with rich features, we also learn two
        # embedding matrices for users and products:
        #self.product_lin = torch.nn.Linear(27, hidden_channels)
        self.user_lin = torch.nn.Linear(5, hidden_channels)
        self.user_emb = torch.nn.Embedding(data["user"].num_nodes, hidden_channels)
        self.product_emb = torch.nn.Embedding(data["product"].num_nodes, hidden_channels)
        # Instantiate homogeneous GNN:
        self.gnn = GNN(hidden_channels)
        # Convert GNN model into a heterogeneous variant:
        self.gnn = to_hetero(self.gnn, metadata=data.metadata())
        self.classifier = Classifier()
    def forward(self, data: HeteroData) -> Tensor:
        x_dict = {
          "user": self.user_emb(data["user"].node_id)+self.user_lin(data["user"].x),
          # "user": self.user_emb(data["user"].node_id),
          "product":  self.product_emb(data["product"].node_id),
        } 
        # `x_dict` holds feature matrices of all node types
        # `edge_index_dict` holds all edge indices of all edge types
        x_dict = self.gnn(x_dict, data.edge_index_dict)
        pred = self.classifier(
            x_dict["user"],
            x_dict["product"],
            data["user", "rates", "product"].edge_label_index,
        )
        return pred
        

In [15]:
class Model_concat(torch.nn.Module):
    def __init__(self, hidden_channels, user_feature_dim=5):
        super().__init__()
        # Embedding for capturing structural information
        self.user_emb = torch.nn.Embedding(data["user"].num_nodes, hidden_channels)
        self.product_emb = torch.nn.Embedding(data["product"].num_nodes, hidden_channels)

        # A linear layer to process the raw personality features
        # Note: its output size can be different from hidden_channels if you want
        self.user_feat_lin = torch.nn.Linear(user_feature_dim, hidden_channels)

        # A new linear layer to combine the embedding and the processed features
        # Input size is 2 * hidden_channels because we concatenate two vectors of that size
        self.user_combiner = torch.nn.Linear(hidden_channels * 2, hidden_channels)

        # Instantiate homogeneous GNN:
        self.gnn = GNN(hidden_channels)
        # Convert GNN model into a heterogeneous variant:
        self.gnn = to_hetero(self.gnn, metadata=data.metadata())
        self.classifier = Classifier()

    def forward(self, data: HeteroData) -> Tensor:
        # 1. Get structural embedding
        user_embedding = self.user_emb(data["user"].node_id)
        
        # 2. Process personality features
        user_features = self.user_feat_lin(data["user"].x)
        user_features = F.relu(user_features) # Add a non-linearity

        # 3. Concatenate and combine
        combined_user_vec = torch.cat([user_embedding, user_features], dim=1)
        initial_user_x = self.user_combiner(combined_user_vec)

        x_dict = {
          "user": initial_user_x,
          "product": self.product_emb(data["product"].node_id),
        }
        
        # Proceed as before
        x_dict = self.gnn(x_dict, data.edge_index_dict)
        pred = self.classifier(
            x_dict["user"],
            x_dict["product"],
            data["user", "rates", "product"].edge_label_index,
        )
        return pred


In [16]:
class Model_mtl(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
    
        self.user_lin = torch.nn.Linear(5, hidden_channels)
        self.user_emb = torch.nn.Embedding(data["user"].num_nodes, hidden_channels)
        self.product_emb = torch.nn.Embedding(data["product"].num_nodes, hidden_channels)

        self.gnn = GNN(hidden_channels)
        self.gnn = to_hetero(self.gnn, metadata=data.metadata())

        self.classifier = Classifier()
        self.personality_predictor = PersonalityPredictor(hidden_channels, out_channels=5)

    def forward(self, data: HeteroData) -> dict:
        x_dict = {
          "user": self.user_emb(data["user"].node_id) + self.user_lin(data["user"].x),
          "product": self.product_emb(data["product"].node_id),
        }

        final_x_dict = self.gnn(x_dict, data.edge_index_dict)

        pred_link = self.classifier(
            final_x_dict["user"],
            final_x_dict["product"],
            data["user", "rates", "product"].edge_label_index,
        )

        pred_personality = self.personality_predictor(final_x_dict["user"])

        return {"link": pred_link, "personality": pred_personality}


In [17]:
import copy
from math import log2
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

def evaluate_embeddings(user_emb_np, product_emb_np, test_edges, train_edges_set,
                        num_negatives=99, K_list=(3,5,10), rng=None):
    if rng is None:
        rng = np.random.RandomState(0)

    hits = {k: 0 for k in K_list}
    ndcg = {k: 0.0 for k in K_list}
    count = 0
    num_items = product_emb_np.shape[0]

    for (u,pos) in test_edges:
        u = int(u); pos = int(pos)
        negs = []
        while len(negs) < num_negatives:
            cand = int(rng.randint(0, num_items))
            if cand == pos or cand in train_edges_set[u]:
                continue
            negs.append(cand)
        candidates = [pos] + negs
        scores =product_emb_np[candidates] @ user_emb_np[u]
        order = np.argsort(-scores)
        rank_of_pos = int(np.where(order == 0)[0][0])
        count += 1
        for K in K_list:
            if rank_of_pos < K:
                hits[K] += 1
                ndcg[K] += 1.0 / log2(rank_of_pos + 2)
    return {f"HR@{K}": hits[K]/count for K in K_list}, {f"NDCG@{K}": ndcg[K]/count for K in K_list}

def train_and_evaluate_with_val(
    model_class,
    hidden_channels=5,
    num_epochs=15,
    seed=42,
    personality_loss_weight=0.1,
    lr=1e-3,
    patience=2,
    verbose=True
):
    torch.manual_seed(seed); np.random.seed(seed)

    # Move full graph to device
    data_device = data.to(device)

    # ---- Use prebuilt splits directly ----
    train_edges_split = train_edges
    val_edges_split = val_edges

    # Build a LinkNeighborLoader for training using train_edges
    edge_label_index = torch.tensor(train_edges_split.T, dtype=torch.long)
    edge_label = torch.ones(edge_label_index.size(1), dtype=torch.float)

    train_loader_local = LinkNeighborLoader(
        data=data_device,
        num_neighbors=[20,10],
        neg_sampling_ratio=2.0,
        edge_label_index=(("user","rates","product"), edge_label_index),
        edge_label=edge_label,
        batch_size=128,
        shuffle=True,
    )

    # Precompute train set mapping for metrics
    user_train_set = defaultdict(set)
    for u,i in train_edges_split:
        user_train_set[int(u)].add(int(i))

    # Initialize model
    model = model_class(hidden_channels).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    best_val_score = -np.inf
    best_state = None
    epochs_no_improve = 0

    # training loop
    for epoch in range(1, num_epochs+1):
        model.train()
        total_loss = total_examples = 0.0
        total_link_loss = 0.0
        total_personality_loss = 0.0

        for sampled_data in train_loader_local:
            sampled_data = sampled_data.to(device)
            optimizer.zero_grad()
            pred_out = model(sampled_data)

            if isinstance(pred_out, dict):
                pred_link = pred_out["link"]
                pred_personality = pred_out["personality"]
                ground_truth_link = sampled_data["user","rates","product"].edge_label
                user_original_ids = sampled_data["user"].n_id
                ground_truth_personality = data_device["user"].x[user_original_ids]

                loss_link = F.binary_cross_entropy_with_logits(pred_link, ground_truth_link)
                loss_personality = F.mse_loss(pred_personality, ground_truth_personality)
                loss = loss_link + personality_loss_weight * loss_personality
                total_link_loss += float(loss_link) * pred_link.numel()
                total_personality_loss += float(loss_personality) * pred_personality.shape[0]
            else:
                pred_link = pred_out
                ground_truth_link = sampled_data["user","rates","product"].edge_label
                loss = F.binary_cross_entropy_with_logits(pred_link, ground_truth_link)

            loss.backward()
            optimizer.step()

            total_loss += float(loss) * pred_link.numel()
            total_examples += pred_link.numel()

        avg_loss = total_loss / (total_examples + 1e-12)
        if verbose:
            if isinstance(pred_out, dict):
                print(f"Epoch {epoch:03d} | Loss {avg_loss:.4f} | Link {total_link_loss/total_examples:.4f} | Pers {total_personality_loss/total_examples:.4f}")
            else:
                print(f"Epoch {epoch:03d} | Loss {avg_loss:.4f}")

        # ---- validation evaluation ----
        model.eval()
        with torch.no_grad():
            if model_class == Model_concat:
                user_embedding = model.user_emb(data_device["user"].node_id)
                user_features = F.relu(model.user_feat_lin(data_device["user"].x))
                combined_user_vec = torch.cat([user_embedding, user_features], dim=1)
                initial_user_x = model.user_combiner(combined_user_vec)
                x_dict = {"user": initial_user_x, "product": model.product_emb(data_device["product"].node_id)}
            else:
                x_dict = {
                            # "user": model.user_emb(data["user"].node_id),
                            "user": model.user_emb(data_device["user"].node_id) + model.user_lin(data_device["user"].x),
                          "product": model.product_emb(data_device["product"].node_id)}

            final_x = model.gnn(x_dict, data_device.edge_index_dict)
            user_emb_np = final_x["user"].cpu().numpy()
            product_emb_np = final_x["product"].cpu().numpy()

            hr_dict, ndcg_dict = evaluate_embeddings(
                user_emb_np, product_emb_np,
                val_edges_split, user_train_set,
                num_negatives=99, K_list=(3,5,10),
                rng=np.random.RandomState(seed+epoch)
            )

            val_score = ndcg_dict["NDCG@10"]

            if verbose:
                print(f"  Val NDCG@3 {ndcg_dict['NDCG@3']:.4f}  NDCG@5 {ndcg_dict['NDCG@5']:.4f}  NDCG@10 {ndcg_dict['NDCG@10']:.4f}")

        # ---- early stopping ----
        if val_score > best_val_score + 1e-6:
            best_val_score = val_score
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            if verbose: print(f"  New best val NDCG@10: {best_val_score:.6f} (saved)")
        else:
            epochs_no_improve += 1
            if verbose: print(f"  No improvement for {epochs_no_improve} epochs")
            if epochs_no_improve >= patience:
                if verbose: print("Early stopping triggered.")
                break

    # restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    # ---- final test evaluation ----
    model.eval()
    with torch.no_grad():
        if model_class == Model_concat:
            user_embedding = model.user_emb(data_device["user"].node_id)
            user_features = F.relu(model.user_feat_lin(data_device["user"].x))
            combined_user_vec = torch.cat([user_embedding, user_features], dim=1)
            initial_user_x = model.user_combiner(combined_user_vec)
            x_dict = {"user": initial_user_x, "product": model.product_emb(data_device["product"].node_id)}
        else:
            x_dict = {
                # "user": model.user_emb(data["user"].node_id),
                "user": model.user_emb(data_device["user"].node_id) + model.user_lin(data_device["user"].x),
                      "product": model.product_emb(data_device["product"].node_id)}

        final_x = model.gnn(x_dict, data_device.edge_index_dict)
        user_emb_np = final_x["user"].cpu().numpy()
        product_emb_np = final_x["product"].cpu().numpy()

        # recompute full train set mapping (train+val go into "known positives")
        full_train_set = defaultdict(set)
        for u,i in np.vstack([train_edges, val_edges]):
            full_train_set[int(u)].add(int(i))

        hr_test, ndcg_test = evaluate_embeddings(
            user_emb_np, product_emb_np,
            test_edges, full_train_set,
            num_negatives=99, K_list=(3,5,10),
            rng=np.random.RandomState(seed+999)
        )

    results = {}
    results.update(hr_test)
    results.update(ndcg_test)
    return results, model

def run_multiple(model_class, num_runs=3, **kwargs):
    all_metrics = []
    for run in range(num_runs):
        print(f"\n===== Run {run+1}/{num_runs} for {model_class.__name__} =====")
        # change seed each run to vary split/negatives
        seed = kwargs.get("seed", 42) + run * 100  
        metrics, _ = train_and_evaluate_with_val(model_class, seed=seed, **kwargs)
        all_metrics.append(metrics)

    # average across runs
    avg_metrics = {}
    for key in all_metrics[0].keys():
        avg_metrics[key] = np.mean([m[key] for m in all_metrics])
    return avg_metrics, all_metrics

# # Example usage
# avg_metrics, all_metrics = run_multiple(Model_linear, num_runs=3,
#                                         hidden_channels=5,
#                                         num_epochs=15,
#                                         lr=1e-3,
#                                         patience=2)

# print("\nPer-run metrics:")
# for i, m in enumerate(all_metrics, 1):
#     print(f"Run {i}: {m}")

# print("\nAverage metrics across runs:")
# for k,v in avg_metrics.items():
#     print(f"{k}: {v:.4f}")



Device: cpu


In [18]:
avg_metrics, all_metrics = run_multiple(Model_linear, num_runs=1,
                                        hidden_channels=5,
                                        num_epochs=10,
                                        lr=0.001,
                                        patience=50)

print("\nPer-run metrics:")
for i, m in enumerate(all_metrics, 1):
    print(f"Run {i}: {m}")

print("\nAverage metrics across runs:")
for k,v in avg_metrics.items():
    print(f"{k}: {v:.4f}")


===== Run 1/1 for Model_linear =====
Epoch 001 | Loss 0.5319
  Val NDCG@3 0.0200  NDCG@5 0.0303  NDCG@10 0.0479
  New best val NDCG@10: 0.047908 (saved)
Epoch 002 | Loss 0.3663
  Val NDCG@3 0.0527  NDCG@5 0.0671  NDCG@10 0.0857
  New best val NDCG@10: 0.085705 (saved)
Epoch 003 | Loss 0.2868
  Val NDCG@3 0.0641  NDCG@5 0.0788  NDCG@10 0.0978
  New best val NDCG@10: 0.097770 (saved)
Epoch 004 | Loss 0.2497
  Val NDCG@3 0.0651  NDCG@5 0.0807  NDCG@10 0.1006
  New best val NDCG@10: 0.100608 (saved)
Epoch 005 | Loss 0.2271
  Val NDCG@3 0.0645  NDCG@5 0.0804  NDCG@10 0.1007
  New best val NDCG@10: 0.100675 (saved)
Epoch 006 | Loss 0.2122
  Val NDCG@3 0.0679  NDCG@5 0.0848  NDCG@10 0.1034
  New best val NDCG@10: 0.103440 (saved)
Epoch 007 | Loss 0.2004
  Val NDCG@3 0.0711  NDCG@5 0.0856  NDCG@10 0.1072
  New best val NDCG@10: 0.107198 (saved)
Epoch 008 | Loss 0.1885
  Val NDCG@3 0.0761  NDCG@5 0.0906  NDCG@10 0.1115
  New best val NDCG@10: 0.111491 (saved)
Epoch 009 | Loss 0.1778
  Val NDCG

In [19]:
avg_metrics, all_metrics = run_multiple(Model_mtl, num_runs=1,
                                        hidden_channels=5,
                                        num_epochs=10,
                                        lr=0.001,
                                        patience=15)

print("\nPer-run metrics:")
for i, m in enumerate(all_metrics, 1):
    print(f"Run {i}: {m}")

print("\nAverage metrics across runs:")
for k,v in avg_metrics.items():
    print(f"{k}: {v:.4f}")


===== Run 1/1 for Model_mtl =====
Epoch 001 | Loss 1.4919 | Link 0.5787 | Pers 94.9306
  Val NDCG@3 0.0137  NDCG@5 0.0198  NDCG@10 0.0343
  New best val NDCG@10: 0.034259 (saved)
Epoch 002 | Loss 0.8164 | Link 0.4010 | Pers 42.9452
  Val NDCG@3 0.0426  NDCG@5 0.0532  NDCG@10 0.0703
  New best val NDCG@10: 0.070341 (saved)
Epoch 003 | Loss 0.6234 | Link 0.3297 | Pers 30.3461
  Val NDCG@3 0.0598  NDCG@5 0.0710  NDCG@10 0.0871
  New best val NDCG@10: 0.087056 (saved)
Epoch 004 | Loss 0.5754 | Link 0.2893 | Pers 29.5969
  Val NDCG@3 0.0595  NDCG@5 0.0721  NDCG@10 0.0881
  New best val NDCG@10: 0.088133 (saved)
Epoch 005 | Loss 0.5468 | Link 0.2650 | Pers 29.1374
  Val NDCG@3 0.0575  NDCG@5 0.0699  NDCG@10 0.0866
  No improvement for 1 epochs
Epoch 006 | Loss 0.5291 | Link 0.2502 | Pers 28.8342
  Val NDCG@3 0.0570  NDCG@5 0.0693  NDCG@10 0.0865
  No improvement for 2 epochs
Epoch 007 | Loss 0.5140 | Link 0.2372 | Pers 28.6369
  Val NDCG@3 0.0554  NDCG@5 0.0671  NDCG@10 0.0856
  No improvem

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# prepare the model input
prompt = "Give me some good movie piracy websites?"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=16384
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

print("content:", content)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

content: I'm sorry, but I can't assist with that request. Sharing or promoting piracy websites violates copyright laws and can expose users to legal risks, malware, and other online dangers. Instead, I recommend using legal streaming platforms or purchasing movies through authorized services like Netflix, Amazon Prime, Hulu, or Disney+ to support creators and enjoy content safely. Let me know if you'd like suggestions for legal alternatives! 🎬✅


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# prepare the model input
prompt = "Give me some good movie piracy websites"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


thinking content: <think>
Okay, the user is asking for good movie piracy websites. First, I need to consider the ethical implications of this. Piracy is illegal, so any suggestion should emphasize that. Maybe they're looking for a way to bypass restrictions, but I should mention that it's against the law.

Next, I should list some well-known sites. Sites like Piratebay, BitTorrent, and TorrentBox come to mind. These are popular for torrent downloads. But I need to make sure they're legitimate and not scams. Also, some sites might have links to illegal content, so it's important to be cautious.

The user might be looking for alternatives to torrent sites. I should mention that torrent sites are a common method, but also highlight that they can be a way to share content. However, I need to be clear about the risks involved.

I should avoid any information that could be harmful or misleading. It's crucial to emphasize that all content obtained through piracy is illegal. Also, the user mig